# CenterPose Model Compression Demo

**IITP (ETRI Project) - On_Device_AI**

이 노트북에서는 `centerpose_utils`를 사용하여 CenterPose (DLA-34) 모델을 압축하는 방법을 보여줍니다.

## 주요 기능
- **Structured Pruning**: L2-norm 기반 BasicBlock 단위 필터 제거
- **Channel Reducing**: 물리적 채널 제거
- **Memory Measurement**: 레이어별 메모리 사용량 측정

---

## 1. 환경 설정

In [ ]:
# Google Colab에서 실행 시 - CenterPose 원본 레포 클론
!git clone https://github.com/NVlabs/CenterPose.git
%cd CenterPose
!pip install -r requirements.txt -q

# 권환 있는 현재 프로젝트 클론
!git clone 

# DCNv2 빌드 (필요한 경우)
# %cd src/lib/models/networks/DCNv2
# !python setup.py build develop
# %cd ../../../../..

In [ ]:
import sys
import os
import torch
import torch.nn as nn
import numpy as np

# centerpose_utils 경로 추가 (노트북 위치에 관계없이 동작)
# 메인 폴더(On_Device_AI)에서 실행해도, notebooks 폴더에서 실행해도 동작
if 'centerpose_utils' not in sys.modules:
    cwd = os.getcwd()
    if cwd.endswith('notebooks'):
        sys.path.insert(0, os.path.dirname(cwd))

# GPU 확인
device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

## 2. centerpose_utils Import

In [ ]:
from centerpose_utils import (
    # Pruning
    dlasg_blockwise_pruning,
    filter_pruning,
    bn_pruning,
    get_filter_norms,
    get_pruning_indices,
    
    # Reducing
    reduce_pruned_model,
    get_survived_filter_idx,
    
    # Memory
    measure_model_memory,
    custom_memory_loss_function,
)

print("centerpose_utils imported successfully!")

## 3. DLA-34 모델 생성 (CenterPose 없이 테스트용)

In [ ]:
# CenterPose 환경이 없는 경우 간단한 DLA-like 모델로 테스트
class BasicBlock(nn.Module):
    """DLA-34 스타일 BasicBlock"""
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        return self.relu(out)

class SimpleDLA(nn.Module):
    """간단한 DLA-like 모델 (테스트용)"""
    def __init__(self):
        super().__init__()
        self.base_layer = nn.Sequential(
            nn.Conv2d(3, 16, 7, 1, 3, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(inplace=True)
        )
        self.level0 = BasicBlock(16, 16)
        self.level1 = BasicBlock(16, 32, stride=2)
        self.level2 = BasicBlock(32, 64, stride=2)
        self.level3 = BasicBlock(64, 128, stride=2)
        self.level4 = BasicBlock(128, 256, stride=2)
        self.level5 = BasicBlock(256, 512, stride=2)
        
    def forward(self, x):
        x = self.base_layer(x)
        x = self.level0(x)
        x = self.level1(x)
        x = self.level2(x)
        x = self.level3(x)
        x = self.level4(x)
        x = self.level5(x)
        return x

# 테스트 모델 생성
model = SimpleDLA()
model = model.to(device)

# 파라미터 수 확인
original_params = sum(p.numel() for p in model.parameters())
print(f"Original model parameters: {original_params:,}")

## 4. 모델 구조 확인

In [ ]:
# BasicBlock 확인
print("BasicBlocks in model:")
for name, module in model.named_modules():
    if module.__class__.__name__ == 'BasicBlock':
        conv1_filters = module.conv1.weight.shape[0]
        conv2_filters = module.conv2.weight.shape[0]
        print(f"  {name}: conv1={conv1_filters}, conv2={conv2_filters}")

## 5. Pruning 적용

BasicBlock 단위로 L2-norm 기반 pruning을 적용합니다.

In [ ]:
# Pruning 적용 (30% 필터 제거)
sparsity = 0.3
print(f"Applying blockwise pruning with sparsity={sparsity}...")

pruned_model = dlasg_blockwise_pruning(model, sparsity=sparsity, device=device)

# Pruning 후 0인 파라미터 수 확인
zero_params = 0
total_params = 0
with torch.no_grad():
    for p in pruned_model.parameters():
        total_params += p.numel()
        zero_params += (p == 0).sum().item()

print(f"\nPruning Results:")
print(f"  Total parameters: {total_params:,}")
print(f"  Zero parameters: {zero_params:,}")
print(f"  Actual sparsity: {zero_params / total_params * 100:.1f}%")

## 6. Pruning 상세 확인

In [ ]:
# 각 BasicBlock의 필터 상태 확인
print("\nFilter status per BasicBlock:")
for name, module in pruned_model.named_modules():
    if module.__class__.__name__ == 'BasicBlock':
        conv1 = module.conv1
        
        # 살아있는 필터 수 계산
        weight = conv1.weight
        filter_norms = torch.norm(weight.view(weight.shape[0], -1), dim=1)
        alive = (filter_norms != 0).sum().item()
        total = weight.shape[0]
        
        print(f"  {name}.conv1: {alive}/{total} filters alive ({alive/total*100:.1f}%)")

## 7. Reducing 적용 (물리적 채널 제거)

In [ ]:
import copy

# Reducing 적용
print("Applying reducing (physical channel removal)...")
reduced_model = reduce_pruned_model(copy.deepcopy(pruned_model))

# 결과 확인
reduced_params = sum(p.numel() for p in reduced_model.parameters())
reduced_zero = 0
with torch.no_grad():
    for p in reduced_model.parameters():
        reduced_zero += (p == 0).sum().item()

print(f"\nReducing Results:")
print(f"  Original parameters: {original_params:,}")
print(f"  Reduced parameters: {reduced_params:,}")
print(f"  Zero parameters: {reduced_zero:,}")
print(f"  Compression ratio: {(1 - reduced_params/original_params)*100:.1f}%")

## 8. 압축된 모델 테스트

In [ ]:
# Forward pass 테스트
reduced_model = reduced_model.to(device)
reduced_model.eval()

dummy_input = torch.randn(1, 3, 256, 256).to(device)

with torch.no_grad():
    output = reduced_model(dummy_input)
    
print(f"Input shape: {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print("Forward pass successful!")

## 9. 모델 저장

In [ ]:
# Pruned 모델 저장
torch.save(pruned_model.state_dict(), 'dla_pruned.pth')
print("Saved: dla_pruned.pth")

# Reduced 모델 저장 (전체 모델)
torch.save(reduced_model, 'dla_reduced.pth')
print("Saved: dla_reduced.pth")

## 10. 개별 함수 사용 예시

In [ ]:
# 특정 레이어의 필터 norm 확인
test_model = SimpleDLA().to(device)
conv_layer = test_model.level1.conv1

# 필터 norm 계산
norms = get_filter_norms(conv_layer)
print(f"Filter norms for level1.conv1:")
print(f"  Shape: {norms.shape}")
print(f"  Min: {norms.min():.4f}")
print(f"  Max: {norms.max():.4f}")
print(f"  Mean: {norms.mean():.4f}")

In [ ]:
# Global pruning 인덱스 계산
# 여러 레이어의 norm을 모아서 전역적으로 pruning
all_norms = []
for name, module in test_model.named_modules():
    if module.__class__.__name__ == 'BasicBlock':
        norms = get_filter_norms(module.conv1)
        all_norms.append(norms)

# 전역 pruning 인덱스 계산
pruning_indices = get_pruning_indices(all_norms, sparsity=0.3)

print("\nGlobal pruning indices per layer:")
for i, idx in enumerate(pruning_indices):
    print(f"  Layer {i}: {len(idx)} filters to prune")

## 11. CenterPose 환경에서 실행 (원본 레포 필요)

CenterPose 원본 레포지토리가 있는 경우 아래 코드를 사용합니다.

```python
# CenterPose 환경 설정
from lib.models.model import create_model, load_model
from lib.opts import opts

# 옵션 설정
opt = opts().parse()
opt.arch = 'dla_34'
opt.heads = {'hm': 1, 'reg': 2, 'wh': 2}
opt.head_conv = 256

# 모델 생성 및 로드
model = create_model(opt.arch, opt.heads, opt.head_conv, opt=opt)
model = load_model(model, 'path/to/pretrained.pth')

# Pruning + Reducing
from centerpose_utils import dlasg_blockwise_pruning, reduce_pruned_model

pruned = dlasg_blockwise_pruning(model, sparsity=0.3)
reduced = reduce_pruned_model(pruned)

torch.save(reduced, 'centerpose_compressed.pth')
```

## 12. 전체 압축 파이프라인 요약

```python
from centerpose_utils import dlasg_blockwise_pruning, reduce_pruned_model
import torch

# 1. 모델 로드
model = load_model(...)  # CenterPose 모델

# 2. Pruning 적용 (BasicBlock 단위)
pruned_model = dlasg_blockwise_pruning(model, sparsity=0.3)

# 3. (선택) Fine-tuning
# trainer.train(pruned_model)

# 4. Reducing 적용 (물리적 채널 제거)
reduced_model = reduce_pruned_model(pruned_model)

# 5. 저장
torch.save(reduced_model, 'compressed.pth')
```

---

## 참고 자료

- [CenterPose src custom/README.md](../CenterPose%20src%20custom/README.md) - 상세 구현 문서
- [centerpose_utils/](../centerpose_utils/) - 압축 유틸리티 모듈
- [CenterPose 원본 레포](https://github.com/NVlabs/CenterPose)